<a href="https://colab.research.google.com/github/sirisha142007/Prompt-Engineering-AI-Productivity/blob/main/AI_Chat_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai gradio

In [ ]:
import os
import gradio as gr
from openai import OpenAI
from getpass import getpass
import json

In [ ]:
# Enter your OpenAI API Key securely

api_key = getpass("Enter your OpenAI API Key: ")

# Store API key in environment variable
os.environ["OPENAI_API_KEY"] = api_key

# Connect to OpenAI
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

print("API Authentication Successful!")

In [ ]:
# Model configuration

MODEL_NAME = "gpt-4o-mini"

TEMPERATURE = 0.7
MAX_TOKENS = 500

SYSTEM_PROMPT = """
You are a helpful AI assistant.
Answer questions clearly and accurately.
Use simple language and provide examples when needed.
"""

print("Model:", MODEL_NAME)
print("Temperature:", TEMPERATURE)
print("Maximum tokens:", MAX_TOKENS)

In [ ]:
# Initialize conversation history

chat_history = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

print("Conversation history initialized!")
print(chat_history)

In [ ]:
def generate_response(user_message, history):

    try:
        if not user_message.strip():
            return "Please enter a message."

        # Add user message
        history.append({
            "role": "user",
            "content": user_message
        })

        # Send conversation to LLM
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=history,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        # Extract AI response
        ai_response = response.choices[0].message.content

        # Save assistant response
        history.append({
            "role": "assistant",
            "content": ai_response
        })

        # Display token usage
        usage = response.usage

        print("Input Tokens:", usage.prompt_tokens)
        print("Output Tokens:", usage.completion_tokens)
        print("Total Tokens:", usage.total_tokens)

        return ai_response

    except Exception as e:
        return f"Error occurred: {str(e)}"

In [ ]:
def generate_structured_response(question):

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "Return the answer as valid JSON with keys: topic, answer, and key_points."
                },
                {
                    "role": "user",
                    "content": question
                }
            ],
            response_format={"type": "json_object"},
            max_tokens=500
        )

        result = response.choices[0].message.content

        # Convert JSON string into Python dictionary
        structured_data = json.loads(result)

        return json.dumps(
            structured_data,
            indent=4
        )

    except Exception as e:
        return f"Error: {str(e)}"


# Test structured response
print(generate_structured_response(
    "Explain Artificial Intelligence"
))

In [ ]:
# Track token usage

total_input_tokens = 0
total_output_tokens = 0

def track_usage(input_tokens, output_tokens):

    global total_input_tokens
    global total_output_tokens

    total_input_tokens += input_tokens
    total_output_tokens += output_tokens

    print("Total Input Tokens:", total_input_tokens)
    print("Total Output Tokens:", total_output_tokens)
    print("Total Tokens Used:",
          total_input_tokens + total_output_tokens)

In [ ]:
import gradio as gr

def chat_with_ai(message, history, instructions):

    try:
        if not message.strip():
            return history, ""

        # Apply custom instructions
        system_message = {
            "role": "system",
            "content": instructions if instructions.strip()
            else SYSTEM_PROMPT
        }

        # Prepare conversation messages
        messages = [system_message]

        # Include previous conversation
        for item in history:
            if item["role"] in ["user", "assistant"]:
                messages.append(item)

        messages.append({
            "role": "user",
            "content": message
        })

        # Get AI response
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        answer = response.choices[0].message.content

        # Track API usage
        track_usage(
            response.usage.prompt_tokens,
            response.usage.completion_tokens
        )

        # Update conversation history
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer}
        ]

        return history, ""

    except Exception as e:
        error_message = f"Error: {str(e)}"

        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": error_message}
        ]

        return history, ""


# Create application interface

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "# 🤖 AI Chat Application"
    )

    gr.Markdown(
        "Chat with an AI assistant powered by OpenAI."
    )

    chatbot = gr.Chatbot(
        label="Conversation",
        height=400,
        type="messages"
    )

    history_state = gr.State([])

    instructions = gr.Textbox(
        label="Custom Instructions",
        value=SYSTEM_PROMPT,
        placeholder="Enter your custom instructions..."
    )

    message = gr.Textbox(
        label="Your Message",
        placeholder="Ask me anything..."
    )

    with gr.Row():

        send = gr.Button(
            "Send",
            variant="primary"
        )

        clear = gr.Button("Clear Chat")

    send.click(
        chat_with_ai,
        inputs=[message, history_state, instructions],
        outputs=[chatbot, message]
    ).then(
        lambda: [],
        outputs=history_state
    )

    message.submit(
        chat_with_ai,
        inputs=[message, history_state, instructions],
        outputs=[chatbot, message]
    ).then(
        lambda: [],
        outputs=history_state
    )

    clear.click(
        lambda: ([], []),
        outputs=[chatbot, history_state]
    )

# Launch application
demo.launch(share=True)

In [ ]:
import gradio as gr

def chat_with_ai(message, history, instructions):

    if not message.strip():
        return history, history, ""

    try:
        # Set custom instructions
        system_message = {
            "role": "system",
            "content": instructions if instructions.strip()
            else SYSTEM_PROMPT
        }

        # Prepare messages
        messages = [system_message] + history

        messages.append({
            "role": "user",
            "content": message
        })

        # Call OpenAI API
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        answer = response.choices[0].message.content

        # Track tokens
        track_usage(
            response.usage.prompt_tokens,
            response.usage.completion_tokens
        )

        # Update history
        updated_history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer}
        ]

        return updated_history, updated_history, ""

    except Exception as e:

        error = f"Something went wrong: {str(e)}"

        updated_history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": error}
        ]

        return updated_history, updated_history, ""


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🤖 AI Chat Application")
    gr.Markdown("Powered by OpenAI API and Python")

    chatbot = gr.Chatbot(
        label="Chat History",
        height=400,
        type="messages"
    )

    history_state = gr.State([])

    instructions = gr.Textbox(
        label="Custom Instructions",
        value=SYSTEM_PROMPT
    )

    message = gr.Textbox(
        label="Enter Your Message",
        placeholder="Ask your question here..."
    )

    with gr.Row():
        send = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear Chat")

    send.click(
        chat_with_ai,
        inputs=[message, history_state, instructions],
        outputs=[chatbot, history_state, message]
    )

    message.submit(
        chat_with_ai,
        inputs=[message, history_state, instructions],
        outputs=[chatbot, history_state, message]
    )

    clear.click(
        lambda: ([], []),
        outputs=[chatbot, history_state]
    )

demo.launch(share=True)